# 01: Pull GTFS, OSM, & Census Data for the Comparison-City Relocations

Pulls what's needed to compute walk+transit isochrones via r5py for the "did-relocate" validation set:
NWSL teams that moved stadiums in the last few years, used to test whether the
accessibility-augmented attendance model actually predicts what happened when a team's
transit access changed.

Old & new stadium pairs (from `nwsl-project/data/processed/stadiums.csv`, already geocoded):

| Team | Old stadium | Years | New stadium | Years |
|---|---|---|---|---|
| Kansas City Current | Children's Mercy Park (Kansas City, **KS**) | 2022-2023 | CPKC Stadium (Kansas City, **MO**) | 2024-2026 |
| San Diego Wave FC | Torero Stadium (San Diego, CA) | 2022 | Snapdragon Stadium (San Diego, CA) | 2023-2026 |
| Seattle Reign FC | Cheney Stadium (Tacoma, WA) | 2019-2021 | Lumen Field (Seattle, WA) | 2022-2026 |
| Washington Spirit | Maryland SoccerPlex (Boyds, **MD**) | 2016-2019 | Audi Field (Washington, **DC**) | 2021-2026 |

**Note**: Current, Spirit, and Gotham each straddle a state/district line, so both sides get pulled. 
Sources pulled here, same pattern as `08_gtfs_data.ipynb`:
1. GTFS static feeds per metro (transit agency schedules)
2. OSM street extracts (Geofabrik, for walk-to-stop routing legs)
3. Census tract boundaries (TIGER/Line, free) + ACS population (needs `CENSUS_API_KEY`)
4. The relocation reference table itself


In [1]:
import os
import time
import requests
import pandas as pd

DATA_PROCESSED_DIR = os.path.join("..", "data", "processed")
GTFS_DIR = os.path.join("..", "data", "raw", "gtfs")
OSM_DIR = os.path.join("..", "data", "raw", "osm")
CENSUS_DIR = os.path.join("..", "data", "raw", "census")
for d in (GTFS_DIR, OSM_DIR, CENSUS_DIR):
    os.makedirs(d, exist_ok=True)


def download_file(url, dest_path, timeout=180, headers=None, retries=3):
    """Streams url to dest_path. Skips re-downloading if the file already exists
    (with a nonzero size) which makes this notebook resumable/idempotent on re-run.
    Downloads to a .part sibling and renames on success only, so a dropped
    connection never leaves a truncated file that looks "already downloaded"."""
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        print(f"{os.path.basename(dest_path)}: already downloaded, skipping")
        return
    tmp_path = dest_path + ".part"
    for attempt in range(1, retries + 1):
        try:
            print(f"Downloading {os.path.basename(dest_path)} (attempt {attempt}/{retries}) ...")
            with requests.get(url, stream=True, timeout=timeout, headers=headers) as r:
                r.raise_for_status()
                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            os.replace(tmp_path, dest_path)
            print(f"  -> {os.path.getsize(dest_path) / 1e6:.1f} MB")
            return
        except (requests.exceptions.RequestException, OSError) as e:
            print(f"  attempt {attempt} failed: {e}")
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            if attempt == retries:
                raise
            time.sleep(5 * attempt)


## GTFS static feeds

One feed per transit agency serving each metro. All checked live against the actual servers
before writing this cell (200 status, real zip content-type) except WMATA, which requires a
free API key (`https://developer.wmata.com/`, instant signup, no approval wait). Set
`WMATA_API_KEY` in your environment and re-run; the cell below skips if it's absent,
same pattern as the Census key below.


In [2]:
GTFS_FEEDS = {
    # Kansas City KCATA (RideKC) covers both the KS and MO sides of the metro, incl. streetcar.
    # Pulled from the Mobility Database because more reliable
    "kcata.zip": "https://files.mobilitydatabase.org/mdb-187/mdb-187-202607240150/mdb-187-202607240150.zip",
    # San Diego MTS covers bus + trolley
    "sdmts.zip": "http://www.sdmts.com/google_transit_files/google_transit.zip",
    # Seattle/Tacoma King County Metro (Seattle/Lumen Field side), Sound Transit rail
    # (Link light rail + Sounder), Pierce Transit (Tacoma/Cheney Stadium side)
    "king_county_metro.zip": "https://www.soundtransit.org/GTFS-KCM/google_transit.zip",
    "sound_transit_rail.zip": "https://www.soundtransit.org/GTFS-rail/40_gtfs.zip",
    "pierce_transit.zip": "https://www.soundtransit.org/GTFS-PT/gtfs.zip",
}

for name, url in GTFS_FEEDS.items():
    download_file(url, os.path.join(GTFS_DIR, name))

# WMATA (DC metrorail + metrobus) needs a free API key
WMATA_API_KEY = os.environ.get("WMATA_API_KEY")
if not WMATA_API_KEY:
    print(
        "\nNo WMATA_API_KEY found in the environment -- skipping WMATA bus/rail GTFS.\n"
        "Sign up (instant, no approval wait) at https://developer.wmata.com/, then:\n"
        "  export WMATA_API_KEY='your_key_here'\n"
        "and re-run this cell. Without it, the DC side of the Washington Spirit isochrones\n"
        "will only reflect walking, not transit."
    )
else:
    wmata_headers = {"api_key": WMATA_API_KEY}
    download_file(
        "https://api.wmata.com/gtfs/bus-gtfs-static.zip",
        os.path.join(GTFS_DIR, "wmata_bus.zip"),
        headers=wmata_headers,
    )
    download_file(
        "https://api.wmata.com/gtfs/rail-gtfs-static.zip",
        os.path.join(GTFS_DIR, "wmata_rail.zip"),
        headers=wmata_headers,
    )


kcata.zip: already downloaded, skipping
sdmts.zip: already downloaded, skipping
king_county_metro.zip: already downloaded, skipping
sound_transit_rail.zip: already downloaded, skipping
pierce_transit.zip: already downloaded, skipping
wmata_bus.zip: already downloaded, skipping
wmata_rail.zip: already downloaded, skipping


## OSM street network extracts (for walk-to-stop routing legs)

Regional `.osm.pbf` extracts from Geofabrik, one per state touched by an old or new stadium:
MO + KS (Kansas City), CA (San Diego), WA (Seattle/Tacoma), MD + DC (Washington). These are
large so `download_file`'s skip-if-exists behavior means this
only actually downloads once per state, and states are shared across metros where relevant.


In [3]:
OSM_EXTRACTS = {
    "missouri-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/missouri-latest.osm.pbf",
    "kansas-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/kansas-latest.osm.pbf",
    "california-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/california-latest.osm.pbf",
    "washington-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/washington-latest.osm.pbf",
    "maryland-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/maryland-latest.osm.pbf",
    "district-of-columbia-latest.osm.pbf": "https://download.geofabrik.de/north-america/us/district-of-columbia-latest.osm.pbf",
}

for name, url in OSM_EXTRACTS.items():
    download_file(url, os.path.join(OSM_DIR, name), timeout=900)


missouri-latest.osm.pbf: already downloaded, skipping
kansas-latest.osm.pbf: already downloaded, skipping
california-latest.osm.pbf: already downloaded, skipping
washington-latest.osm.pbf: already downloaded, skipping
maryland-latest.osm.pbf: already downloaded, skipping
district-of-columbia-latest.osm.pbf: already downloaded, skipping


### Merge into per-metro combined extracts

Same reason as `08_gtfs_data.ipynb`: `r5py`'s `TransportNetwork` takes a single OSM file, but
KC and DC each span two jurisdictions. San Diego and Seattle/Tacoma are single-state and don't
need merging. 


In [2]:
import shutil
import subprocess

MERGES = {
    "kc-mo-ks-merged.osm.pbf": ["missouri-latest.osm.pbf", "kansas-latest.osm.pbf"],
    "dc-md-merged.osm.pbf": ["district-of-columbia-latest.osm.pbf", "maryland-latest.osm.pbf"],
}

if shutil.which("osmium") is None:
    print("osmium CLI not found -> install with `brew install osmium-tool`, then re-run this cell.")
else:
    for merged_name, inputs in MERGES.items():
        merged_path = os.path.join(OSM_DIR, merged_name)
        if os.path.exists(merged_path) and os.path.getsize(merged_path) > 0:
            print(f"{merged_name}: already merged ({os.path.getsize(merged_path) / 1e6:.1f} MB), skipping")
            continue
        input_paths = [os.path.join(OSM_DIR, f) for f in inputs]
        subprocess.run(
            ["osmium", "merge", *input_paths, "-o", merged_path, "--overwrite"],
            check=True,
        )
        print(f"{merged_name}: merged ({os.path.getsize(merged_path) / 1e6:.1f} MB)")


kc-mo-ks-merged.osm.pbf: already merged (308.6 MB), skipping
dc-md-merged.osm.pbf: already merged (234.3 MB), skipping


### Clip to per-metro bounding boxes

Need to clip networks with `osmium extract` to avoid wasting time. Create a bounding box around stadiums.
Bounding boxes are stadium-pair extent + a ~15-20 mile buffer (generous enough that a 60-min
transit isochrone from either stadium won't run off the edge of the extract).


In [5]:
METRO_BBOXES = {
    # name: (source osm file, bbox: min_lon, min_lat, max_lon, max_lat)
    "san-diego-metro.osm.pbf": ("california-latest.osm.pbf", (-117.35, 32.60, -116.90, 33.00)),
    "seattle-tacoma-metro.osm.pbf": ("washington-latest.osm.pbf", (-122.65, 47.05, -122.15, 47.75)),
    "kansas-city-metro.osm.pbf": ("kc-mo-ks-merged.osm.pbf", (-95.10, 38.90, -94.30, 39.35)),
    "dc-metro.osm.pbf": ("dc-md-merged.osm.pbf", (-77.55, 38.65, -76.85, 39.35)),
}

if shutil.which("osmium") is None:
    print("osmium CLI not found -- install with `brew install osmium-tool`, then re-run this cell.")
else:
    for clipped_name, (source_name, bbox) in METRO_BBOXES.items():
        clipped_path = os.path.join(OSM_DIR, clipped_name)
        if os.path.exists(clipped_path) and os.path.getsize(clipped_path) > 0:
            print(f"{clipped_name}: already clipped ({os.path.getsize(clipped_path) / 1e6:.1f} MB), skipping")
            continue
        source_path = os.path.join(OSM_DIR, source_name)
        bbox_str = ",".join(str(v) for v in bbox)
        subprocess.run(
            ["osmium", "extract", "--bbox", bbox_str, "-o", clipped_path, source_path, "-O"],
            check=True,
        )
        print(f"{clipped_name}: clipped from {source_name} ({os.path.getsize(clipped_path) / 1e6:.1f} MB)")


san-diego-metro.osm.pbf: already clipped (44.8 MB), skipping
seattle-tacoma-metro.osm.pbf: already clipped (83.4 MB), skipping
kansas-city-metro.osm.pbf: already clipped (41.6 MB), skipping
dc-metro.osm.pbf: already clipped (71.5 MB), skipping


## Census tract boundaries (free, no key) + population by tract (needs `CENSUS_API_KEY`)

TIGER/Line tract *boundaries* are public with no key and downloaded unconditionally below. The
ACS population attribute needs a free key -> same skip pattern as `08_gtfs_data.ipynb`:
if `CENSUS_API_KEY` isn't set, this prints setup instructions and moves on. State FIPS codes:
MO=29, KS=20, CA=06, WA=53, MD=24, DC=11.


In [6]:
TIGER_TRACTS = {
    "tl_2023_29_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_29_tract.zip",  # MO
    "tl_2023_20_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_20_tract.zip",  # KS
    "tl_2023_06_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_06_tract.zip",  # CA
    "tl_2023_53_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_53_tract.zip",  # WA
    "tl_2023_24_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_24_tract.zip",  # MD
    "tl_2023_11_tract.zip": "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_11_tract.zip",  # DC
}
for name, url in TIGER_TRACTS.items():
    download_file(url, os.path.join(CENSUS_DIR, name))


tl_2023_29_tract.zip: already downloaded, skipping
tl_2023_20_tract.zip: already downloaded, skipping
tl_2023_06_tract.zip: already downloaded, skipping
tl_2023_53_tract.zip: already downloaded, skipping
tl_2023_24_tract.zip: already downloaded, skipping
tl_2023_11_tract.zip: already downloaded, skipping


In [7]:
CENSUS_STATE_FIPS = {"MO": "29", "KS": "20", "CA": "06", "WA": "53", "MD": "24", "DC": "11"}
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")
acs_path = os.path.join(CENSUS_DIR, "acs_population_by_tract_comparison_cities.csv")

if not CENSUS_API_KEY:
    print(
        "No CENSUS_API_KEY found in the environment -- skipping ACS population pull.\n"
        "Sign up (usually instant, no approval wait) at "
        "https://api.census.gov/data/key_signup.html, then:\n"
        "  export CENSUS_API_KEY='your_key_here'\n"
        "and re-run this cell. Tract *boundaries* were already downloaded above regardless.\n"
        "03_accessibility_demographics.ipynb will skip population-weighting until this exists."
    )
else:
    frames = []
    for abbr, fips in CENSUS_STATE_FIPS.items():
        url = (
            "https://api.census.gov/data/2023/acs/acs5"
            f"?get=NAME,B01003_001E&for=tract:*&in=state:{fips}&key={CENSUS_API_KEY}"
        )
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        rows = resp.json()
        df = pd.DataFrame(rows[1:], columns=rows[0])
        df["state_abbr"] = abbr
        frames.append(df)
        time.sleep(0.5)
    acs = pd.concat(frames, ignore_index=True).rename(columns={"B01003_001E": "population"})
    acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
    acs.to_csv(acs_path, index=False)
    print(f"Wrote {acs_path} ({len(acs)} tracts)")


Wrote ../data/raw/census/acs_population_by_tract_comparison_cities.csv (15077 tracts)


## Relocation reference table

Old vs. new stadium for each of the 4 comparison-city moves, pulled directly from
`nwsl-project/data/processed/stadiums.csv`.


In [8]:
NWSL_PROJECT_PROCESSED = os.path.join("..", "..", "..", "QSS 20", "nwsl-project", "data", "processed")
stadiums = pd.read_csv(os.path.join(NWSL_PROJECT_PROCESSED, "stadiums.csv"))

RELOCATIONS = [
    # team, stadium_id, label, name, years
    ("Kansas City Current", "NPqxy6XQ9d", "old", "Children's Mercy Park", "2022-2023"),
    ("Kansas City Current", "xW5p3L0Mg1", "new", "CPKC Stadium", "2024-2026"),
    ("San Diego Wave FC", "0Oq62v7q6D", "old", "Torero Stadium", "2022"),
    ("San Diego Wave FC", "Oa5wKXY514", "new", "Snapdragon Stadium", "2023-2026"),
    ("Seattle Reign FC", "Oa5wdz9q14", "old", "Cheney Stadium", "2019-2021"),
    ("Seattle Reign FC", "9Yqda07QvJ", "new", "Lumen Field", "2022-2026"),
    ("Washington Spirit", "vzqoGO7qap", "old", "Maryland SoccerPlex", "2016-2019"),
    ("Washington Spirit", "xW5pwORMg1", "new", "Audi Field", "2021-2026"),
]

rows = []
for team, stadium_id, label, name, years in RELOCATIONS:
    match = stadiums.loc[stadiums["stadium_id"] == stadium_id]
    if match.empty:
        print(f"WARNING: stadium_id {stadium_id} ({name}) not found in stadiums.csv")
        continue
    r = match.iloc[0]
    rows.append({
        "team": team,
        "label": label,
        "stadium_id": stadium_id,
        "name": name,
        "years": years,
        "latitude": r["latitude"],
        "longitude": r["longitude"],
    })

relocation_stadiums = pd.DataFrame(rows)
relocation_stadiums.to_csv(os.path.join(DATA_PROCESSED_DIR, "comparison_relocation_stadiums.csv"), index=False)
relocation_stadiums


,team,label,stadium_id,name,years,latitude,longitude
0,Kansas City Current,old,NPqxy6XQ9d,Children's Mercy Park,2022-2023,39.121400,-94.821000
1,Kansas City Current,new,xW5p3L0Mg1,CPKC Stadium,2024-2026,39.119379,-94.566591
2,San Diego Wave FC,old,0Oq62v7q6D,Torero Stadium,2022,32.797200,-117.170800
3,San Diego Wave FC,new,Oa5wKXY514,Snapdragon Stadium,2023-2026,32.784242,-117.122390
4,Seattle Reign FC,old,Oa5wdz9q14,Cheney Stadium,2019-2021,47.238314,-122.497602
5,Seattle Reign FC,new,9Yqda07QvJ,Lumen Field,2022-2026,47.595200,-122.331600
6,Washington Spirit,old,vzqoGO7qap,Maryland SoccerPlex,2016-2019,39.152800,-77.313600
7,Washington Spirit,new,xW5pwORMg1,Audi Field,2021-2026,38.868300,-77.012200


## Manifest

In [9]:
for label, d in [("GTFS feeds", GTFS_DIR), ("OSM extracts", OSM_DIR), ("Census", CENSUS_DIR)]:
    print(f"{label}:")
    for f in sorted(os.listdir(d)):
        path = os.path.join(d, f)
        if os.path.isfile(path):
            print(f"  {f}: {os.path.getsize(path) / 1e6:.1f} MB")
    print()


GTFS feeds:
  kcata.zip: 1.8 MB
  king_county_metro.zip: 17.5 MB
  pierce_transit.zip: 5.1 MB
  sdmts.zip: 4.5 MB
  sound_transit_rail.zip: 1.6 MB
  wmata_bus.zip: 49.8 MB
  wmata_rail.zip: 2.5 MB

OSM extracts:
  california-latest.osm.pbf: 1325.0 MB
  dc-md-merged.osm.pbf: 234.3 MB
  dc-metro.osm.pbf: 71.5 MB
  district-of-columbia-latest.osm.pbf: 20.9 MB
  kansas-city-metro.osm.pbf: 41.6 MB
  kansas-latest.osm.pbf: 115.1 MB
  kc-mo-ks-merged.osm.pbf: 308.6 MB
  maryland-latest.osm.pbf: 213.7 MB
  missouri-latest.osm.pbf: 193.8 MB
  san-diego-metro.osm.pbf: 44.8 MB
  seattle-tacoma-metro.osm.pbf: 83.4 MB
  washington-latest.osm.pbf: 361.4 MB

Census:
  acs_population_by_tract_comparison_cities.csv: 1.2 MB
  tl_2023_06_tract.zip: 32.5 MB
  tl_2023_11_tract.zip: 0.4 MB
  tl_2023_20_tract.zip: 3.1 MB
  tl_2023_24_tract.zip: 5.8 MB
  tl_2023_29_tract.zip: 10.5 MB
  tl_2023_53_tract.zip: 9.9 MB

